# Extended Data Fig. 5a–b: knowledge-base coverage

## Purpose

Plot cumulative literature counts (a) and disciplinary coverage (b).

## Inputs and provenance

Panel a reads the tracked `Phytomni-PaperYear-for_plot.csv`: 126 years (1900–2025), totaling 3,964,798 records. Panel b retains the author-supplied inline table: 30 named disciplines plus Others, totaling 2,879,834 classified records. These are distinct source-table scopes; neither table is rescaled to the manuscript-wide inventory. The year-count file was renamed without changing its bytes from `paper_meta_match_v1_v2_250810_year_counts.csv`.

## Run and outputs

Run all cells with the standard `ir` kernel in this directory. Set `PHYTOMNI_SAVE=1` to write `output/extended_data_fig.5a.pdf` and `output/extended_data_fig.5b.pdf` (14 × 10 inches). The default run displays plots without writing figures.

In [ ]:
save_figs <- Sys.getenv("PHYTOMNI_SAVE") == "1"
if (save_figs) dir.create("output", showWarnings = FALSE)

# Extended Data Fig. 5a & 5b
# Load all necessary packages

library(tidyverse)
library(scales)
library(RColorBrewer)
library(treemapify)

# Display and export size in inches.
options(repr.plot.width = 14, repr.plot.height = 10)

In [ ]:
df_raw <- read_csv("Phytomni-PaperYear-for_plot.csv")

In [ ]:
# Data processing for cumulative line plot (Fig. 5a)
# Create full time series from 1900 to 2025 and compute cumulative count
df_full <- tibble(PY = 1900:2025) %>%
  left_join(df_raw, by = "PY") %>%
  mutate(count = replace_na(count, 0)) %>%
  arrange(PY) %>%
  mutate(cum_count = cumsum(count))

# Select data points at 10-year intervals (and include 2025)
df_bin_cumulative <- df_full %>%
  filter(PY %% 10 == 0 | PY == 2025) %>%
  arrange(PY)

# Show processed data
print(df_bin_cumulative, n = nrow(df_bin_cumulative))


In [ ]:
# Cumulative count line plot (Fig. 5a)
# Extended Data Fig. 5a: Cumulative count of papers over time
p_5a <- ggplot(df_bin_cumulative, aes(x = PY, y = cum_count, group = 1)) +
  geom_line(color = "#0072B2", linewidth = 1) +
  geom_point(color = "#0072B2", size = 2) +
  scale_x_continuous(
    breaks = df_bin_cumulative$PY,
    labels = df_bin_cumulative$PY,
    guide = guide_axis(angle = 45)
  ) +
  scale_y_continuous(
    labels = label_number(big.mark = " ", accuracy = 1),
    expand = expansion(mult = c(0, 0.05))
  ) +
  labs(
    title = "Cumulative Count of Papers (1900-2025)",
    x = "Year",
    y = "Cumulative Count"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    panel.border = element_rect(colour = "black", fill = NA, linewidth = 1),
    plot.background = element_rect(colour = "black", fill = NA, linewidth = 1),
    plot.title = element_text(hjust = 0.5, face = "bold"),
    axis.title = element_text(face = "bold"),
    panel.grid.major = element_line(linetype = "dashed", color = "grey80")
  )

p_5a
if (save_figs) ggsave("output/extended_data_fig.5a.pdf", plot = p_5a, width = 14, height = 10)

In [ ]:
# Data import and preview (Fig. 5b)


type_counts <- data.frame(
  analysis_col = c(
    "Biochemistry & Molecular Biology",
    "Others",
    "Agriculture",
    "Plant Sciences",
    "Environmental Sciences & Ecology",
    "Computer Science",
    "Science & Technology",
    "Biotechnology & Applied Microbiology",
    "Chemistry",
    "Cell Biology",
    "Oncology",
    "Genetics & Heredity",
    "Engineering",
    "Microbiology",
    "Food Science & Technology",
    "Life Sciences & Biomedicine",
    "Fisheries",
    "Zoology",
    "Pharmacology & Pharmacy",
    "Biodiversity & Conservation",
    "Immunology",
    "Forestry",
    "Marine & Freshwater Biology",
    "Neurosciences & Neurology",
    "Endocrinology & Metabolism",
    "Virology",
    "General & Internal Medicine",
    "Developmental Biology",
    "Cardiovascular System & Cardiology",
    "Veterinary Sciences",
    "Research & Experimental Medicine"
  ),
  n = c(434551, 325683, 278378, 194820, 188274, 169278, 154612,
        99546, 94570, 86799, 82167L, 78194, 75017, 74744,
        57013, 48711, 46837, 41656L, 41138, 38048, 36866,
        32614, 31770, 27025, 26149L, 24794, 19070, 18468,
        18213, 17554, 17275),
  percentage = c(0.150894461, 0.113090893, 0.096664599, 0.067649733,
                 0.065376685, 0.058780471, 0.053687817, 0.034566576,
                 0.032838698, 0.030140279, 0.028531853, 0.027152259,
                 0.026049071, 0.025954274, 0.019797322, 0.016914517,
                 0.016263785, 0.014464723, 0.014284851, 0.013211873,
                 0.012801432, 0.011324958, 0.011031886, 0.009384221,
                 0.009080037, 0.008609524, 0.006621909, 0.006412870,
                 0.006324323, 0.006095490, 0.005998610),
  stringsAsFactors = FALSE
)

In [ ]:
head(type_counts)

In [ ]:
# Treemap of disciplinary coverage (Fig. 5b)

# Extended Data Fig. 5b: Disciplinary coverage as a treemap

n_types <- nrow(type_counts)
colors <- colorRampPalette(brewer.pal(min(n_types, 12), "Set3"))(n_types)

# Adjust legend layout based on number of categories
legend_cols <- ifelse(n_types > 30, 2, 1)
legend_font_size <- ifelse(n_types > 30, 10, 12)

p_5b <- ggplot(type_counts, aes(area = n, fill = analysis_col)) +
  geom_treemap(
    color = "white",
    size = 1.5,
    alpha = 0.9,
    layout = "squarified"
  ) +
  scale_fill_manual(
    name = "",
    values = colors,
    guide = guide_legend(
      ncol = legend_cols,
      keyheight = unit(0.4, "cm"),
      keywidth = unit(0.5, "cm")
    )
  ) +
  labs(
    title = "Disciplinary Coverage"
  ) +
  theme_void() +
  theme(
    plot.title = element_text(
      size = 24,
      face = "bold",
      hjust = 0.5,
      margin = margin(b = 20)
    ),
    plot.subtitle = element_text(
      size = 16,
      hjust = 0.5,
      margin = margin(b = 30)
    ),
    legend.position = "bottom",
    legend.box = "vertical",
    legend.title = element_text(
      size = 16,
      face = "bold",
      margin = margin(b = 10)
    ),
    legend.text = element_text(
      size = legend_font_size,
      margin = margin(r = 15)
    ),
    legend.margin = margin(t = 20, b = 20),
    plot.margin = margin(2, 2, 2, 2, "cm")
  )

p_5b
if (save_figs) ggsave("output/extended_data_fig.5b.pdf", plot = p_5b, width = 14, height = 10)